In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pyspark

In [2]:
spark = SparkSession.builder \
    .appName("Understand caching") \
    .master("local[*]") \
    .config("spark.executor.memory", "512M") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 16:17:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
_schema = """
    transacted_at string,
    trx_id string,
    retailer_id string,
    description string,
    amount float,
    city_id string
"""


In [10]:
df = spark.read.format("csv").schema(_schema).option("header", True).load("/Users/AnhHuynh/Documents/PySpark/new_sales.txt")

In [12]:
df.where("amount > 300").show()

+--------------------+----------+-----------+--------------------+-------+----------+
|       transacted_at|    trx_id|retailer_id|         description| amount|   city_id|
+--------------------+----------+-----------+--------------------+-------+----------+
|2017-11-24T19:00:...|1734117022|  847200066|Wal-Mart  ppd id:...|1737.26|1646415505|
|2017-11-24T19:00:...|1734117030| 1953761884|Home Depot     pp...|  384.5| 287177635|
|2017-11-24T19:00:...|1734117153|  847200066|unkn        Kings...|2907.57|1483931123|
|2017-11-24T19:00:...|1734117241|  486576507|              iTunes|2912.67|1663872965|
|2017-11-24T19:00:...|2076947146|  511877722|unkn     ccd id: ...|1915.35|1698762556|
|2017-11-24T19:00:...|2076947113| 1996661856|AutoZone  arc id:...| 1523.6|1759612211|
|2017-11-24T19:00:...|2076946994| 1898522855|Target    ppd id:...|2589.93|2074005445|
|2017-11-24T19:00:...|2076946121|  562903918|unkn    ccd id: 5...| 315.86|1773943669|
|2017-11-24T19:00:...|2076946063| 1070485878|Amazon.co

In [17]:
# Cached dataframe

"""
For cache, we need an action, and count and write -- these are preferred as they will scan 
the whole dataset resulting in proper caching
"""

# df.cache().count()

df_cache = df.cache()

* Default storage level for cache in MEMORY_AND_DISK for dataframes and datasets

In [18]:
df_cache.count()

7202569

* The count() action touches every partition, ensuring the entire DataFrame is cached.

In [19]:
df_cache.where("amount > 300").show()

# After caching, the query returned much faster because spark scan in-memory and disk data

+--------------------+----------+-----------+--------------------+-------+----------+
|       transacted_at|    trx_id|retailer_id|         description| amount|   city_id|
+--------------------+----------+-----------+--------------------+-------+----------+
|2017-11-24T19:00:...|1734117022|  847200066|Wal-Mart  ppd id:...|1737.26|1646415505|
|2017-11-24T19:00:...|1734117030| 1953761884|Home Depot     pp...|  384.5| 287177635|
|2017-11-24T19:00:...|1734117153|  847200066|unkn        Kings...|2907.57|1483931123|
|2017-11-24T19:00:...|1734117241|  486576507|              iTunes|2912.67|1663872965|
|2017-11-24T19:00:...|2076947146|  511877722|unkn     ccd id: ...|1915.35|1698762556|
|2017-11-24T19:00:...|2076947113| 1996661856|AutoZone  arc id:...| 1523.6|1759612211|
|2017-11-24T19:00:...|2076946994| 1898522855|Target    ppd id:...|2589.93|2074005445|
|2017-11-24T19:00:...|2076946121|  562903918|unkn    ccd id: 5...| 315.86|1773943669|
|2017-11-24T19:00:...|2076946063| 1070485878|Amazon.co

In [16]:
# Remove cache
df.unpersist()

DataFrame[transacted_at: string, trx_id: string, retailer_id: string, description: string, amount: float, city_id: string]

* Use **cache()** when:
    * You want to keep the DataFrame in memory.
    * The dataset fits comfortably in memory.
    * You're reusing the DataFrame multiple times.

* Use **persist()** when you need control over how Spark stores the data.

In [22]:
df_persist = df.persist(pyspark.StorageLevel.MEMORY_ONLY)

In [23]:
df_persist.write.format("noop").mode("overwrite").save()

In [21]:
spark.catalog.clearCache()